# Bitcoin Nibble-Bundle Beam Search

A richer admissible side bundle for real Bitcoin headers: staged carry bits, full carry-mask patterns, and the nibble-wise Hamming silhouette of the `h` register.

In [1]:

from __future__ import annotations
from dataclasses import dataclass
import hashlib
import random
import statistics
import struct
from typing import Dict, List, Tuple

MASK32 = 0xFFFFFFFF

H0 = [
    0x6A09E667, 0xBB67AE85, 0x3C6EF372, 0xA54FF53A,
    0x510E527F, 0x9B05688C, 0x1F83D9AB, 0x5BE0CD19,
]

K = [
    0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5, 0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,
    0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3, 0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,
    0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc, 0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,
    0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7, 0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,
    0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13, 0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,
    0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3, 0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,
    0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5, 0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,
    0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208, 0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2,
]

def u32(x: int) -> int:
    return x & MASK32

def rotr(x: int, n: int) -> int:
    x &= MASK32
    return ((x >> n) | (x << (32 - n))) & MASK32

def ch(x: int, y: int, z: int) -> int:
    return ((x & y) ^ (~x & z)) & MASK32

def maj(x: int, y: int, z: int) -> int:
    return ((x & y) ^ (x & z) ^ (y & z)) & MASK32

def Sigma0(x: int) -> int:
    return rotr(x, 2) ^ rotr(x, 13) ^ rotr(x, 22)

def Sigma1(x: int) -> int:
    return rotr(x, 6) ^ rotr(x, 11) ^ rotr(x, 25)

def sigma0(x: int) -> int:
    return rotr(x, 7) ^ rotr(x, 18) ^ (x >> 3)

def sigma1(x: int) -> int:
    return rotr(x, 17) ^ rotr(x, 19) ^ (x >> 10)

def hw(x: int) -> int:
    return (x & MASK32).bit_count()

def nibble_hws(x: int) -> Tuple[int, ...]:
    return tuple(hw((x >> shift) & 0xF) for shift in range(0, 32, 4))

def add32(a: int, b: int) -> Tuple[int, int]:
    total = (a & MASK32) + (b & MASK32)
    return total & MASK32, int(total >> 32)

def carry_mask_add(a: int, b: int) -> int:
    a &= MASK32
    b &= MASK32
    carry = a & b
    union = carry
    s = a ^ b
    while carry:
        carry = (carry << 1) & MASK32
        newcarry = s & carry
        union |= newcarry
        s ^= carry
        carry = newcarry
    return union

def words_from_block(block: bytes) -> List[int]:
    return list(struct.unpack(">16I", block))

def expand_schedule(w16: List[int]) -> List[int]:
    W = list(w16)
    for t in range(16, 64):
        W.append(u32(sigma1(W[t - 2]) + W[t - 7] + sigma0(W[t - 15]) + W[t - 16]))
    return W

def pad_sha256(msg: bytes) -> bytes:
    bit_len = len(msg) * 8
    out = msg + b"\x80"
    while len(out) % 64 != 56:
        out += b"\x00"
    out += struct.pack(">Q", bit_len)
    return out

def dbl_sha256_hex_display(msg: bytes) -> str:
    return hashlib.sha256(hashlib.sha256(msg).digest()).digest()[::-1].hex()

@dataclass
class RoundTraceFull:
    t: int
    a: int
    b: int
    c: int
    d: int
    e: int
    f: int
    g: int
    h: int
    Wt: int
    T1: int
    T2: int
    stage_carries: Tuple[int, int, int, int]
    stage_masks: Tuple[int, int, int, int]
    h_hw: int
    h_nibble_hw: Tuple[int, ...]

def compress_block_trace_full(block: bytes, state: List[int]) -> Dict[str, object]:
    W = expand_schedule(words_from_block(block))
    a, b, c, d, e, f, g, h = state
    traces: List[RoundTraceFull] = []
    for t in range(64):
        s1 = Sigma1(e)
        chv = ch(e, f, g)
        sA, c1 = add32(h, s1)
        sB, c2 = add32(sA, chv)
        sC, c3 = add32(sB, K[t])
        T1, c4 = add32(sC, W[t])
        m1 = carry_mask_add(h, s1)
        m2 = carry_mask_add(sA, chv)
        m3 = carry_mask_add(sB, K[t])
        m4 = carry_mask_add(sC, W[t])
        T2 = u32(Sigma0(a) + maj(a, b, c))
        traces.append(
            RoundTraceFull(
                t=t, a=a, b=b, c=c, d=d, e=e, f=f, g=g, h=h,
                Wt=W[t], T1=T1, T2=T2,
                stage_carries=(c1, c2, c3, c4),
                stage_masks=(m1, m2, m3, m4),
                h_hw=hw(h),
                h_nibble_hw=nibble_hws(h),
            )
        )
        new_a = u32(T1 + T2)
        new_e = u32(d + T1)
        a, b, c, d, e, f, g, h = new_a, a, b, c, new_e, e, f, g
    state_out = [u32(state[i] + v) for i, v in enumerate([a, b, c, d, e, f, g, h])]
    return {"W": W, "traces": traces, "working_final": [a, b, c, d, e, f, g, h], "state_out": state_out}

def sha256_trace_full(msg: bytes) -> List[Dict[str, object]]:
    padded = pad_sha256(msg)
    blocks = [padded[i:i+64] for i in range(0, len(padded), 64)]
    state = H0[:]
    out = []
    for idx, block in enumerate(blocks):
        step = compress_block_trace_full(block, state)
        out.append({"block_index": idx, "block": block, "init_state": state[:], **step})
        state = step["state_out"]
    return out

def reverse_step_from_next(next_state: List[int], t: int, W_guess: int) -> Dict[str, object]:
    a1, b1, c1, d1, e1, f1, g1, h1 = next_state
    a_t = b1
    b_t = c1
    c_t = d1
    e_t = f1
    f_t = g1
    g_t = h1
    T2 = u32(Sigma0(a_t) + maj(a_t, b_t, c_t))
    T1 = u32(a1 - T2)
    d_t = u32(e1 - T1)
    s1 = Sigma1(e_t)
    chv = ch(e_t, f_t, g_t)
    const_tail = u32(s1 + chv + K[t])
    h_t = u32(T1 - const_tail - W_guess)
    sA, c1 = add32(h_t, s1)
    sB, c2 = add32(sA, chv)
    sC, c3 = add32(sB, K[t])
    T1_check, c4 = add32(sC, W_guess)
    return {"state": [a_t, b_t, c_t, d_t, e_t, f_t, g_t, h_t], "stage_carries": (c1, c2, c3, c4), "T1_check": T1_check, "T1": T1, "T2": T2}

def geometry_score_nibble(traces: List[RoundTraceFull], next_state: List[int], t: int, guess: int) -> Dict[str, int]:
    obs = traces[t]
    pred = reverse_step_from_next(next_state, t, guess)
    a_t, b_t, c_t, d_t, e_t, f_t, g_t, h_t = pred["state"]
    s1 = Sigma1(e_t)
    chv = ch(e_t, f_t, g_t)
    sA, _ = add32(h_t, s1)
    sB, _ = add32(sA, chv)
    sC, _ = add32(sB, K[t])
    pred_masks = (
        carry_mask_add(h_t, s1),
        carry_mask_add(sA, chv),
        carry_mask_add(sB, K[t]),
        carry_mask_add(sC, guess),
    )
    carry_mismatch = sum(int(a != b) for a, b in zip(pred["stage_carries"], obs.stage_carries))
    mask_bit_mismatch = sum(hw(a ^ b) for a, b in zip(pred_masks, obs.stage_masks))
    h_hw_error = abs(hw(h_t) - obs.h_hw)
    h_nibble_error = sum(abs(a - b) for a, b in zip(nibble_hws(h_t), obs.h_nibble_hw))
    total = 5 * carry_mismatch + mask_bit_mismatch + h_hw_error + h_nibble_error
    return {"score": total, "carry_mismatch": carry_mismatch, "mask_bit_mismatch": mask_bit_mismatch, "h_hw_error": h_hw_error, "h_nibble_error": h_nibble_error}

def as_hex32(x: int) -> str:
    return f"0x{x:08x}"

def nibble_descent(block: Dict[str, object], next_state: List[int], t: int, start: int, sweeps: int = 10) -> Tuple[int, int]:
    cur = start & MASK32
    cur_s = geometry_score_nibble(block["traces"], next_state, t, cur)["score"]
    for _ in range(sweeps):
        improved = False
        for pos in range(8):
            shift = pos * 4
            base = cur & ~(0xF << shift)
            best_word, best_score = cur, cur_s
            for nib in range(16):
                cand = base | (nib << shift)
                s = geometry_score_nibble(block["traces"], next_state, t, cand)["score"]
                if s < best_score or (s == best_score and cand < best_word):
                    best_word, best_score = cand, s
            if best_word != cur:
                cur, cur_s = best_word, best_score
                improved = True
        if not improved:
            break
    return cur, cur_s

def local_candidate_pool(block: Dict[str, object], next_state: List[int], t: int, restarts: int = 64, keep: int = 16, sweeps: int = 10, seed: int = 0) -> List[Tuple[int, int]]:
    rng = random.Random(seed)
    best: Dict[int, int] = {}
    starts = [0, 0xFFFFFFFF] + [rng.getrandbits(32) for _ in range(restarts)]
    for start in starts:
        w, s = nibble_descent(block, next_state, t, start, sweeps=sweeps)
        improved = True
        while improved:
            improved = False
            for bit in range(32):
                cand = w ^ (1 << bit)
                sc = geometry_score_nibble(block["traces"], next_state, t, cand)["score"]
                if sc < s or (sc == s and cand < w):
                    w, s = cand, sc
                    improved = True
        best[w] = min(best.get(w, 10**9), s)
    return sorted((s, w) for w, s in best.items())[:keep]

def beam_search_chain(block: Dict[str, object], t_hi: int = 63, t_lo: int = 60, beam_width: int = 16, restarts: int = 64, keep_local: int = 8, seed: int = 0) -> List[Dict[str, object]]:
    beam = [{"guesses": {}, "next_state": block["working_final"], "total": 0, "per_round": []}]
    rng = random.Random(seed)
    for t in range(t_hi, t_lo - 1, -1):
        new = []
        for path in beam:
            rows = local_candidate_pool(block, path["next_state"], t, restarts=restarts, keep=keep_local, seed=rng.randrange(1 << 30))
            for s, w in rows:
                pred = reverse_step_from_next(path["next_state"], t, w)
                new.append({"guesses": {**path["guesses"], t: w}, "next_state": pred["state"], "total": path["total"] + s, "per_round": path["per_round"] + [(t, s, w)]})
        uniq: Dict[Tuple[Tuple[int, int], ...], Dict[str, object]] = {}
        for p in new:
            key = tuple(sorted(p["guesses"].items()))
            if key not in uniq or (p["total"], key) < (uniq[key]["total"], tuple(sorted(uniq[key]["guesses"].items()))):
                uniq[key] = p
        beam = sorted(uniq.values(), key=lambda p: (p["total"], tuple(sorted(p["guesses"].items()))))[:beam_width]
    return beam

GENESIS_HEADER_HEX = (
    "01000000" + "00" * 32 + "3ba3edfd7a7b12b27ac72c3e67768f617fc81bc3888a51323a9fb8aa4b1e5e4a" + "29ab5f49" + "ffff001d" + "1dac2b7c"
)
BLOCK_328734_HEADER_HEX = (
    "02000000"
    "b6ff0b1b1680a2862a30ca44d346d9e8"
    "910d334beb48ca0c0000000000000000"
    "9d10aa52ee949386ca9385695f04ede2"
    "70dda20810decd12bc9b048aaab31471"
    "24d95a54"
    "30c31b18"
    "fe9f0864"
)
REAL_HEADERS = {
    "genesis": {"height": 0, "known_hash": "000000000019d6689c085ae165831e934ff763ae46a2a6c172b3f1b60a8ce26f", "header": bytes.fromhex(GENESIS_HEADER_HEX)},
    "block_328734": {"height": 328734, "known_hash": "000000000000000009a11b3972c8e532fe964de937c9e0096b43814e67af3728", "header": bytes.fromhex(BLOCK_328734_HEADER_HEX)},
}

def analyze_headers() -> List[Dict[str, object]]:
    rows = []
    for name, item in REAL_HEADERS.items():
        calc = dbl_sha256_hex_display(item["header"])
        p1 = sha256_trace_full(item["header"])
        block = p1[1]
        t = 63
        true_word = block["W"][t]
        true_score = geometry_score_nibble(block["traces"], block["working_final"], t, true_word)["score"]
        rng = random.Random(2026)
        random_scores = []
        for _ in range(5000):
            g = rng.getrandbits(32)
            random_scores.append((geometry_score_nibble(block["traces"], block["working_final"], t, g)["score"], g))
        random_scores.sort()
        pool = local_candidate_pool(block, block["working_final"], t, restarts=64, keep=8, seed=2026)
        beam = beam_search_chain(block, t_hi=63, t_lo=60, beam_width=16, restarts=64, keep_local=8, seed=2026)
        true_chain = {tt: block["W"][tt] for tt in range(60, 64)}
        true_rank = next((i + 1 for i, p in enumerate(beam) if p["guesses"] == true_chain), None)
        rows.append({
            "name": name,
            "computed_hash_matches": calc == item["known_hash"],
            "hash": calc,
            "true_W63": as_hex32(true_word),
            "true_W63_score": true_score,
            "best_random_score": random_scores[0][0],
            "median_random_score": statistics.median([s for s, _ in random_scores]),
            "best_local_descent": [(s, as_hex32(w)) for s, w in pool[:5]],
            "beam_top5": [{"rank": i + 1, "total": p["total"], "is_true_chain": p["guesses"] == true_chain, "guesses": {tt: as_hex32(ww) for tt, ww in sorted(p["guesses"].items())}} for i, p in enumerate(beam[:5])],
            "true_chain_rank_in_beam": true_rank,
            "true_chain_hex": {tt: as_hex32(ww) for tt, ww in sorted(true_chain.items())},
        })
    return rows


## Executed output

In [2]:
====================================================================================================
genesis
hash ok: True 000000000019d6689c085ae165831e934ff763ae46a2a6c172b3f1b60a8ce26f
true W63: 0x86b0b8d5 score: 0
best random score: 20 median random: 69.0
best local descent: [(0, '0x86b0b8d5'), (4, '0x86b0b905'), (4, '0x86b0b914'), (7, '0x86a8b915')]
true chain: {60: '0xfd9e331a', 61: '0xd86c39a0', 62: '0x61cdb5cb', 63: '0x86b0b8d5'}
true chain rank in beam: 1
beam top 5:
  {'rank': 1, 'total': 0, 'is_true_chain': True, 'guesses': {60: '0xfd9e331a', 61: '0xd86c39a0', 62: '0x61cdb5cb', 63: '0x86b0b8d5'}}
  {'rank': 2, 'total': 3, 'is_true_chain': False, 'guesses': {60: '0xfd9e32ea', 61: '0xd86c39a0', 62: '0x61cdb5cb', 63: '0x86b0b8d5'}}
  {'rank': 3, 'total': 5, 'is_true_chain': False, 'guesses': {60: '0xfd9dd31a', 61: '0xd86c39a0', 62: '0x61cdb5cb', 63: '0x86b0b8d5'}}
  {'rank': 4, 'total': 8, 'is_true_chain': False, 'guesses': {60: '0xfd9dd2ea', 61: '0xd86c39a0', 62: '0x61cdb5cb', 63: '0x86b0b8d5'}}
  {'rank': 5, 'total': 10, 'is_true_chain': False, 'guesses': {60: '0xc10e331a', 61: '0xd86c39a0', 62: '0x61cdb5cb', 63: '0x86b0b8d5'}}
====================================================================================================
block_328734
hash ok: True 000000000000000009a11b3972c8e532fe964de937c9e0096b43814e67af3728
true W63: 0xa572aedd score: 0
best random score: 18 median random: 57.0
best local descent: [(0, '0xa572aedd'), (3, '0xa572aee0'), (4, '0xa5732edd'), (7, '0xa5732ee0'), (8, '0xa572b0dd')]
true chain: {60: '0x735e832a', 61: '0x6bebedd6', 62: '0xf444e8df', 63: '0xa572aedd'}
true chain rank in beam: 1
beam top 5:
  {'rank': 1, 'total': 0, 'is_true_chain': True, 'guesses': {60: '0x735e832a', 61: '0x6bebedd6', 62: '0xf444e8df', 63: '0xa572aedd'}}
  {'rank': 2, 'total': 4, 'is_true_chain': False, 'guesses': {60: '0x5b5e832a', 61: '0x6bebedd6', 62: '0xec44e8df', 63: '0xa572aedd'}}
  {'rank': 3, 'total': 4, 'is_true_chain': False, 'guesses': {60: '0x735e432a', 61: '0x6bf3add6', 62: '0xf444e8df', 63: '0xa572aedd'}}
  {'rank': 4, 'total': 5, 'is_true_chain': False, 'guesses': {60: '0x6f5e832a', 61: '0x6bebedd6', 62: '0xec44e8df', 63: '0xa572aedd'}}
  {'rank': 5, 'total': 5, 'is_true_chain': False, 'guesses': {60: '0x715e832a', 61: '0x6bebedd6', 62: '0xec44e8df', 63: '0xa572aedd'}}

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (2339499763.py, line 3)